# Sobol Sensitivity Analysis & Bayesian Inverse Problem
## Rock Mechanics: UCS Prediction from 15 Input Parameters

**Two Tasks:**
1. **Sobol Global Sensitivity Analysis** — Rank the 15 input parameters by their influence on PeakUCS
2. **Bayesian Inverse Problem (MCMC)** — Given a target UCS PDF, infer the posterior PDFs of input parameters

**Methodology (aligned with Kumar & Tiwari 2022 paper):**
- Sobol Total-Order Indices via SALib (Saltelli estimator)
- Gaussian Process (Kriging) as the surrogate/response surface
- MCMC via PyMC for Bayesian inversion

In [ ]:
# ============================================================
# CELL 1: IMPORTS & DATA LOADING
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel, RationalQuadratic
from sklearn.metrics import r2_score

# --- Load Data ---
# If running in Colab, replace these paths with your uploaded file paths
TRAIN_PATH = 'UCS_InputsOutputs_E.csv'          # 100-row training set
TEST_PATH  = 'UCS_InputsOutputs_E_Testing.csv'  # 50-row testing set

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
df_train.columns = df_train.columns.str.strip()
df_test.columns  = df_test.columns.str.strip()

TARGET_COLS  = ['PeakUCS', 'YoungsModulus']
INPUT_COLS   = [c for c in df_train.columns if c not in TARGET_COLS]

print(f'Training rows : {len(df_train)}')
print(f'Testing rows  : {len(df_test)}')
print(f'Input features: {INPUT_COLS}')

In [ ]:
# ============================================================
# CELL 2: DATA CLEANING, SCALING & GPR SURROGATE TRAINING
# ============================================================
from sklearn.multioutput import MultiOutputRegressor

X_train_raw = df_train[INPUT_COLS].apply(pd.to_numeric, errors='coerce')
y_train_raw = df_train[TARGET_COLS].apply(pd.to_numeric, errors='coerce')
X_test_raw  = df_test[INPUT_COLS].apply(pd.to_numeric, errors='coerce')
y_test_raw  = df_test[TARGET_COLS].apply(pd.to_numeric, errors='coerce')

# Impute + Scale
imp_X = SimpleImputer(strategy='mean')
X_train = imp_X.fit_transform(X_train_raw)
X_test  = imp_X.transform(X_test_raw)

imp_y = SimpleImputer(strategy='mean')
y_train = imp_y.fit_transform(y_train_raw)
y_test  = imp_y.transform(y_test_raw)

scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train)

# --- GPR Surrogate (same kernel as original notebook) ---
kernel = (ConstantKernel(1.0, (1e-3, 1e4)) *
          Matern(length_scale=1.0, length_scale_bounds=(1e-4, 1e4), nu=1.5) +
          RationalQuadratic(alpha=0.1) +
          WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-10, 1e-2)))

gpr_base = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, random_state=42)
surrogate = MultiOutputRegressor(gpr_base)
print('Training GPR surrogate (may take ~1 min)...')
surrogate.fit(X_train_sc, y_train_sc)

y_pred = scaler_y.inverse_transform(surrogate.predict(X_test_sc))
nse_ucs = r2_score(y_test[:, 0], y_pred[:, 0])
nse_ym  = r2_score(y_test[:, 1], y_pred[:, 1])
print(f'GPR NSE — PeakUCS: {nse_ucs:.4f} | YoungsModulus: {nse_ym:.4f}')

# Convenience: predict UCS only (unscaled)
def predict_ucs(X_raw):
    """X_raw: (N, 15) array in original units. Returns PeakUCS predictions."""
    Xs = scaler_X.transform(X_raw)
    ys = surrogate.predict(Xs)
    return scaler_y.inverse_transform(ys)[:, 0]  # PeakUCS column

---
## PART 1: Sobol Global Sensitivity Analysis

We use the **Saltelli** quasi-random sampling scheme and compute **Total-Order Sobol Indices** (S_Ti) for all 15 input parameters.  
S_Ti captures both direct effects and all interaction effects of parameter Xi on PeakUCS.  
This directly follows the methodology in §2.3 of Kumar & Tiwari (2022).

In [ ]:
# ============================================================
# CELL 3: SOBOL ANALYSIS — PROBLEM DEFINITION
# ============================================================
from SALib.sample import saltelli
from SALib.analyze import sobol

# Define parameter bounds from training data (min–max ranges)
bounds = [[float(X_train[:, i].min()), float(X_train[:, i].max())]
          for i in range(len(INPUT_COLS))]

problem = {
    'num_vars': len(INPUT_COLS),
    'names'   : INPUT_COLS,
    'bounds'  : bounds
}

print('Sobol Problem Definition:')
for name, b in zip(INPUT_COLS, bounds):
    print(f'  {name:35s}: [{b[0]:.3e}, {b[1]:.3e}]')

In [ ]:
# ============================================================
# CELL 4: SOBOL SAMPLING & EVALUATION
# ============================================================
# N=1024 gives (2*N*(D+2)) = 1024*34 = ~34k model evaluations
# Increase N for higher accuracy if compute allows
N_sobol = 1024
print(f'Generating Saltelli samples (N={N_sobol}) → {N_sobol*(len(INPUT_COLS)+2)*2} evaluations...')

param_values = saltelli.sample(problem, N_sobol, calc_second_order=False)
print(f'Sample matrix shape: {param_values.shape}')

# Evaluate GPR surrogate
print('Running surrogate evaluations...')
Y_sobol = predict_ucs(param_values)
print(f'Output range: [{Y_sobol.min():.3e}, {Y_sobol.max():.3e}] Pa')
print('Done!')

In [ ]:
# ============================================================
# CELL 5: SOBOL INDEX COMPUTATION & VISUALIZATION
# ============================================================
Si = sobol.analyze(problem, Y_sobol, calc_second_order=False, print_to_console=False)

sobol_df = pd.DataFrame({
    'Parameter'    : INPUT_COLS,
    'S1'           : Si['S1'],          # First-order index
    'S1_conf'      : Si['S1_conf'],
    'ST'           : Si['ST'],          # Total-order index  ← main ranking metric
    'ST_conf'      : Si['ST_conf'],
    'Interaction'  : Si['ST'] - Si['S1']  # Interaction contribution
}).sort_values('ST', ascending=False).reset_index(drop=True)

print('\n===== SOBOL INDICES — Ranked by Total-Order (ST) =====')
print(sobol_df[['Parameter', 'S1', 'ST', 'Interaction']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Sobol Global Sensitivity Analysis — PeakUCS', fontsize=15, weight='bold')

colors_st = ['#d62728' if v > 0.1 else '#1f77b4' for v in sobol_df['ST']]

# Total-order bar chart
ax = axes[0]
bars = ax.barh(sobol_df['Parameter'][::-1], sobol_df['ST'][::-1],
               xerr=sobol_df['ST_conf'][::-1], color=colors_st[::-1],
               edgecolor='black', linewidth=0.5, capsize=3)
ax.axvline(0.1, color='red', linestyle='--', linewidth=1.2, label='Threshold = 0.1')
ax.set_xlabel('Total-Order Sobol Index (S_T)', fontsize=11)
ax.set_title('Total-Order Indices (S_T)\n(red bars = sensitive parameters)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='x', linestyle=':', alpha=0.5)

# Stacked S1 vs interaction
ax2 = axes[1]
x = np.arange(len(sobol_df))
ax2.bar(x, sobol_df['S1'], label='First-Order (S1)', color='#2196F3', edgecolor='black', linewidth=0.5)
ax2.bar(x, sobol_df['Interaction'].clip(lower=0), bottom=sobol_df['S1'],
        label='Interaction (ST - S1)', color='#FF9800', edgecolor='black', linewidth=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(sobol_df['Parameter'], rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Sobol Index', fontsize=11)
ax2.set_title('First-Order vs Interaction Effects', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('sobol_indices.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nFigure saved: sobol_indices.png')

# Identify sensitive parameters (ST > 0.05)
sensitive = sobol_df[sobol_df['ST'] > 0.05]['Parameter'].tolist()
print(f'\nSensitive parameters (ST > 0.05): {sensitive}')

---
## PART 2: Bayesian Inverse Problem — Input PDFs from UCS PDF

**Goal:** Given a target UCS distribution (e.g., from field measurements or specifications), recover the posterior distributions of the 15 input parameters.

**Framework:**
- **Prior**: Fit distributions to each input parameter from the training data
- **Likelihood**: Use the GPR surrogate to compute `p(UCS | inputs)` — predictions are Gaussian (mean ± std from GPR)
- **Posterior**: Sample via MCMC (`pymc`) → posterior PDFs for all 15 inputs

This is the Bayesian analog of the jackknife/bootstrap approach in the paper — instead of resampling the input data, we **invert** the surrogate to find input distributions consistent with observed UCS.

In [ ]:
# ============================================================
# CELL 6: FIT UCS PDF FROM TRAINING DATA (or use your own)
# ============================================================
from scipy import stats

ucs_data = y_train[:, 0]  # PeakUCS from training set

# Fit candidate distributions and pick best by AIC (as in paper, Eq. 2)
candidates = {
    'Normal'    : stats.norm,
    'Lognormal' : stats.lognorm,
    'Weibull'   : stats.weibull_min,
    'Gamma'     : stats.gamma,
}

aic_results = {}
fit_params  = {}
for name, dist in candidates.items():
    pars = dist.fit(ucs_data)
    log_lik = np.sum(dist.logpdf(ucs_data, *pars))
    k = len(pars)
    aic = -2 * log_lik + 2 * k   # Eq. (2) from paper
    aic_results[name] = aic
    fit_params[name]  = pars

best_dist_name = min(aic_results, key=aic_results.get)
print('UCS Distribution AIC values:')
for k, v in sorted(aic_results.items(), key=lambda x: x[1]):
    marker = ' ← BEST FIT' if k == best_dist_name else ''
    print(f'  {k:12s}: AIC = {v:.2f}{marker}')

ucs_mean = ucs_data.mean()
ucs_std  = ucs_data.std()
print(f'\nUCS Summary: Mean={ucs_mean:.3e} Pa, Std={ucs_std:.3e} Pa, CoV={ucs_std/ucs_mean:.3f}')

# Plot UCS PDF
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ucs_data, bins=20, density=True, alpha=0.5, color='steelblue',
        edgecolor='white', label='Training Data')
x_range = np.linspace(ucs_data.min(), ucs_data.max(), 300)
for name, dist in candidates.items():
    lw = 2.5 if name == best_dist_name else 1.0
    ls = '-' if name == best_dist_name else '--'
    ax.plot(x_range, dist.pdf(x_range, *fit_params[name]),
            linewidth=lw, linestyle=ls, label=f'{name} (AIC={aic_results[name]:.1f})')
ax.set_xlabel('PeakUCS (Pa)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('UCS Probability Density — Fitted Distributions', fontsize=12)
ax.legend(fontsize=8)
ax.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('ucs_pdf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 7: FIT PRIOR DISTRIBUTIONS FOR EACH INPUT PARAMETER
# ============================================================
# For MCMC we need priors on each input. We fit Normal distributions
# to each input from training data (you can use domain knowledge instead).

input_priors = {}  # {name: (mean, std)} in SCALED space
print('Input parameter statistics (original space):')
print(f'{"Parameter":35s} {"Mean":>15s} {"Std":>15s} {"CoV":>8s}')
print('-' * 80)
for i, col in enumerate(INPUT_COLS):
    m = X_train[:, i].mean()
    s = X_train[:, i].std()
    cov = s / abs(m) if m != 0 else 0
    input_priors[col] = (m, s)
    print(f'{col:35s} {m:15.4e} {s:15.4e} {cov:8.3f}')

In [ ]:
# ============================================================
# CELL 8: GPR UNCERTAINTY — Build a differentiable surrogate
# ============================================================
# For MCMC we need the GPR to return mean AND std (uncertainty)
# We use individual GPR estimators from MultiOutputRegressor

gpr_ucs = surrogate.estimators_[0]  # GPR trained for PeakUCS

def predict_ucs_with_std(X_raw):
    """Returns (mean, std) of PeakUCS prediction for raw input array."""
    Xs = scaler_X.transform(np.atleast_2d(X_raw))
    y_sc_mean, y_sc_std = gpr_ucs.predict(Xs, return_std=True)
    # Unscale: only UCS scaler params needed
    ucs_scale_mean = scaler_y.mean_[0]
    ucs_scale_std  = scaler_y.scale_[0]
    pred_mean = y_sc_mean * ucs_scale_std + ucs_scale_mean
    pred_std  = y_sc_std  * ucs_scale_std
    return pred_mean.flatten(), pred_std.flatten()

# Validate
test_m, test_s = predict_ucs_with_std(X_test[:3])
print('GPR with uncertainty (first 3 test samples):')
for i in range(3):
    print(f'  Predicted: {test_m[i]:.3e} ± {test_s[i]:.3e} Pa | Actual: {y_test[i,0]:.3e} Pa')

In [ ]:
# ============================================================
# CELL 9: MCMC BAYESIAN INVERSION
# ============================================================
# Strategy: PyTensor + PyMC custom likelihood
#
# Model:
#   theta_i ~ Normal(prior_mean_i, prior_std_i)  [scaled space]
#   UCS_pred = GPR_surrogate(theta)
#   UCS_obs  ~ Normal(UCS_pred, GPR_uncertainty)
#
# We observe: target UCS = mean of training UCS
# (Replace 'target_ucs' with any value from field measurements)

import pymc as pm
import pytensor.tensor as pt
from pytensor.graph import Apply
from pytensor.graph.op import Op

# Target UCS observation (use the mean of training data; change as needed)
TARGET_UCS = float(ucs_mean)
print(f'Target UCS for inversion: {TARGET_UCS:.4e} Pa')

# PyTensor wrapper for GPR (needed for PyMC)
class GPRPredict(Op):
    """Custom PyTensor Op that wraps the GPR surrogate."""
    itypes = [pt.dvector]
    otypes = [pt.dvector]
    
    def perform(self, node, inputs, outputs):
        x_scaled = inputs[0].reshape(1, -1)
        mean, std = predict_ucs_with_std(scaler_X.inverse_transform(x_scaled))
        outputs[0][0] = np.array([mean[0], std[0]])

gpr_op = GPRPredict()

# Scale the priors into standardised space
prior_means_sc = np.zeros(len(INPUT_COLS))  # In scaled space mean=0
prior_stds_sc  = np.ones(len(INPUT_COLS))   # In scaled space std≈1, widen to 2

print('Building PyMC model...')
with pm.Model() as bayes_model:
    # --- Priors (in scaled space) ---
    theta = pm.Normal('theta',
                      mu    = prior_means_sc,
                      sigma = 2.0 * prior_stds_sc,  # 2× wider than data spread
                      shape = len(INPUT_COLS))
    
    # --- GPR forward pass (custom Op) ---
    gpr_out = gpr_op(theta)
    ucs_pred_mean = gpr_out[0]
    ucs_pred_std  = gpr_out[1] + ucs_std * 0.05  # add 5% floor uncertainty
    
    # --- Likelihood: observed UCS ~ N(GPR_pred_mean, GPR_pred_std) ---
    ucs_obs = pm.Normal('ucs_obs',
                        mu    = ucs_pred_mean,
                        sigma = ucs_pred_std,
                        observed = TARGET_UCS)

print('Model built. Running MCMC (NUTS sampler)...')
with bayes_model:
    trace = pm.sample(
        draws      = 1000,
        tune       = 500,
        chains     = 2,
        target_accept = 0.85,
        progressbar   = True,
        random_seed   = 42
    )

print('MCMC sampling complete!')

In [ ]:
# ============================================================
# CELL 10: POSTERIOR ANALYSIS
# ============================================================
import arviz as az

# Get posterior samples (scaled space) and transform back to original
theta_post_sc = trace.posterior['theta'].values.reshape(-1, len(INPUT_COLS))  # (2000, 15)
theta_post    = scaler_X.inverse_transform(theta_post_sc)  # back to original units

post_df = pd.DataFrame(theta_post, columns=INPUT_COLS)

print('\n===== POSTERIOR STATISTICS OF INPUT PARAMETERS =====')
print(f'{"Parameter":35s} {"Prior Mean":>15s} {"Post Mean":>15s} {"Post Std":>15s} {"Post CoV":>10s}')
print('-' * 90)
for col in INPUT_COLS:
    prior_m = input_priors[col][0]
    post_m  = post_df[col].mean()
    post_s  = post_df[col].std()
    cov = post_s / abs(post_m) if post_m != 0 else 0
    print(f'{col:35s} {prior_m:15.4e} {post_m:15.4e} {post_s:15.4e} {cov:10.3f}')

In [ ]:
# ============================================================
# CELL 11: VISUALISE POSTERIOR PDFs vs PRIOR
# ============================================================
from scipy.stats import gaussian_kde

n_params = len(INPUT_COLS)
ncols = 3
nrows = int(np.ceil(n_params / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
fig.suptitle('Posterior PDFs of Input Parameters\n(Blue=Prior, Red=Posterior)',
             fontsize=14, weight='bold')
axes_flat = axes.flatten()

for i, col in enumerate(INPUT_COLS):
    ax = axes_flat[i]
    prior_m, prior_s = input_priors[col]
    post_samples = post_df[col].values
    
    # Plot prior
    x_range = np.linspace(prior_m - 3*prior_s, prior_m + 3*prior_s, 300)
    prior_pdf = stats.norm.pdf(x_range, prior_m, prior_s)
    ax.plot(x_range, prior_pdf, 'b-', linewidth=1.5, alpha=0.7, label='Prior (Normal)')
    ax.fill_between(x_range, prior_pdf, alpha=0.15, color='blue')
    
    # Plot posterior KDE
    kde = gaussian_kde(post_samples)
    x_post = np.linspace(post_samples.min(), post_samples.max(), 300)
    post_pdf = kde(x_post)
    ax.plot(x_post, post_pdf, 'r-', linewidth=2.0, label='Posterior (MCMC)')
    ax.fill_between(x_post, post_pdf, alpha=0.25, color='red')
    
    # 95% Credible Interval
    ci_lo = np.percentile(post_samples, 2.5)
    ci_hi = np.percentile(post_samples, 97.5)
    ax.axvline(ci_lo, color='darkred', linestyle=':', linewidth=1.0)
    ax.axvline(ci_hi, color='darkred', linestyle=':', linewidth=1.0, label=f'95% CI')
    
    ax.set_title(col, fontsize=9, weight='bold')
    ax.set_xlabel('Value', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(linestyle=':', alpha=0.4)
    ax.legend(fontsize=6)
    ax.ticklabel_format(style='sci', axis='x', scilimits=(-2, 2))

# Hide unused subplots
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('posterior_input_pdfs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: posterior_input_pdfs.png')

In [ ]:
# ============================================================
# CELL 12: POSTERIOR PREDICTIVE CHECK
# ============================================================
# Sample from posterior and push through GPR to get implied UCS distribution

# Take 500 random posterior samples
n_ppc = 500
idx = np.random.choice(len(theta_post), n_ppc, replace=False)
ppc_ucs = predict_ucs(theta_post[idx])

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ucs_data, bins=25, density=True, alpha=0.45, color='steelblue',
        edgecolor='white', label='Observed UCS (Training Data)')
ax.hist(ppc_ucs, bins=25, density=True, alpha=0.45, color='tomato',
        edgecolor='white', label='Posterior Predictive UCS (MCMC)')

# Best-fit distribution on observed
best_dist = candidates[best_dist_name]
x_range = np.linspace(min(ucs_data.min(), ppc_ucs.min()),
                      max(ucs_data.max(), ppc_ucs.max()), 300)
ax.plot(x_range, best_dist.pdf(x_range, *fit_params[best_dist_name]),
        'b--', linewidth=2, label=f'Fitted {best_dist_name} (Observed)')

ax.axvline(TARGET_UCS, color='black', linewidth=2, linestyle='-',
           label=f'Target UCS = {TARGET_UCS:.3e} Pa')
ax.set_xlabel('PeakUCS (Pa)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Posterior Predictive Check\nDoes the MCMC posterior reproduce the target UCS?',
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('posterior_predictive_check.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nObserved UCS: Mean={ucs_data.mean():.4e}, Std={ucs_data.std():.4e}')
print(f'PPC UCS:      Mean={ppc_ucs.mean():.4e}, Std={ppc_ucs.std():.4e}')

In [ ]:
# ============================================================
# CELL 13: COMBINED SUMMARY FIGURE
# ============================================================
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Panel A: Sobol ST
ax_a = fig.add_subplot(gs[0, 0])
top_n = min(10, len(sobol_df))
top = sobol_df.head(top_n)
colors = ['#d62728' if v > 0.1 else '#1f77b4' for v in top['ST']]
ax_a.barh(top['Parameter'][::-1], top['ST'][::-1],
          xerr=top['ST_conf'][::-1], color=colors[::-1],
          edgecolor='black', linewidth=0.5, capsize=3)
ax_a.axvline(0.1, color='red', linestyle='--', linewidth=1)
ax_a.set_xlabel('S_T (Total-Order Sobol Index)')
ax_a.set_title('A: Sobol Sensitivity Ranking\n(Top 10 Parameters)', weight='bold')
ax_a.grid(axis='x', linestyle=':', alpha=0.5)

# Panel B: UCS PDF
ax_b = fig.add_subplot(gs[0, 1])
ax_b.hist(ucs_data, bins=20, density=True, alpha=0.5, color='steelblue', edgecolor='white')
ax_b.plot(x_range, best_dist.pdf(x_range, *fit_params[best_dist_name]),
          'r-', linewidth=2, label=f'Best fit: {best_dist_name}')
ax_b.set_xlabel('PeakUCS (Pa)')
ax_b.set_ylabel('Density')
ax_b.set_title('B: UCS Probability Density\n(Target Distribution)', weight='bold')
ax_b.legend(fontsize=9)
ax_b.grid(linestyle=':', alpha=0.5)

# Panel C: Most sensitive input posterior (highest ST)
top_param = sobol_df.iloc[0]['Parameter']
ax_c = fig.add_subplot(gs[1, 0])
prior_m, prior_s = input_priors[top_param]
post_samples = post_df[top_param].values
x_pr = np.linspace(prior_m - 3*prior_s, prior_m + 3*prior_s, 300)
ax_c.plot(x_pr, stats.norm.pdf(x_pr, prior_m, prior_s), 'b-', lw=1.5, label='Prior')
ax_c.fill_between(x_pr, stats.norm.pdf(x_pr, prior_m, prior_s), alpha=0.15, color='blue')
kde2 = gaussian_kde(post_samples)
x_po = np.linspace(post_samples.min(), post_samples.max(), 300)
ax_c.plot(x_po, kde2(x_po), 'r-', lw=2, label='Posterior (MCMC)')
ax_c.fill_between(x_po, kde2(x_po), alpha=0.25, color='red')
ax_c.set_title(f'C: Posterior of Most Sensitive Input\n({top_param})', weight='bold')
ax_c.set_xlabel('Value'); ax_c.set_ylabel('Density')
ax_c.legend(fontsize=9); ax_c.grid(linestyle=':', alpha=0.5)
ax_c.ticklabel_format(style='sci', axis='x', scilimits=(-2, 2))

# Panel D: Posterior Predictive Check
ax_d = fig.add_subplot(gs[1, 1])
ax_d.hist(ucs_data, bins=20, density=True, alpha=0.45, color='steelblue',
          edgecolor='white', label='Observed')
ax_d.hist(ppc_ucs, bins=20, density=True, alpha=0.45, color='tomato',
          edgecolor='white', label='Posterior Predictive')
ax_d.axvline(TARGET_UCS, color='black', lw=2, linestyle='--', label='Target')
ax_d.set_xlabel('PeakUCS (Pa)'); ax_d.set_ylabel('Density')
ax_d.set_title('D: Posterior Predictive Check\n(MCMC → UCS)', weight='bold')
ax_d.legend(fontsize=9); ax_d.grid(linestyle=':', alpha=0.5)

fig.suptitle('Rock Mechanics — Sobol Sensitivity & Bayesian Inversion Summary',
             fontsize=14, weight='bold', y=1.01)
plt.savefig('summary_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Summary figure saved: summary_figure.png')

In [ ]:
# ============================================================
# CELL 14: CREDIBLE INTERVALS SUMMARY TABLE
# ============================================================
print('\n===== 95% CREDIBLE INTERVALS (Posterior, Original Units) =====')
print(f'{"Parameter":35s} {"Prior Mean":>15s} {"Post Mean":>15s} {"CI 2.5%":>15s} {"CI 97.5%":>15s}')
print('-' * 95)
ci_data = []
for col in INPUT_COLS:
    ps = post_df[col].values
    pm_val = ps.mean()
    ci_lo  = np.percentile(ps, 2.5)
    ci_hi  = np.percentile(ps, 97.5)
    prior_m = input_priors[col][0]
    print(f'{col:35s} {prior_m:15.4e} {pm_val:15.4e} {ci_lo:15.4e} {ci_hi:15.4e}')
    ci_data.append({'Parameter': col, 'Prior_Mean': prior_m, 'Post_Mean': pm_val,
                    'CI_2.5': ci_lo, 'CI_97.5': ci_hi,
                    'Sobol_ST': sobol_df.set_index('Parameter').loc[col, 'ST']})

ci_df = pd.DataFrame(ci_data).sort_values('Sobol_ST', ascending=False)
ci_df.to_csv('posterior_ci_summary.csv', index=False)
print('\nSaved: posterior_ci_summary.csv')

print('\n===== FINAL INTERPRETATION =====')
print(f'Sensitive parameters (ST > 0.1): {sobol_df[sobol_df["ST"] > 0.1]["Parameter"].tolist()}')
print('These parameters showed the largest shift from prior to posterior,')
print('meaning the observed UCS distribution is most informative about them.')